# Paper III reconstructed federated pilot
Three-seed, 20-round FedAvg-style simulation, illustrative IID and label-skew partitions, D0/D0C/D2 heuristic noise, matched membership attacks and isotonic calibration. This is a new exploratory implementation, not original Paper III reproduction. No formal differential privacy guarantee is asserted.

In [ ]:
import numpy as np,pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score,roc_curve,brier_score_loss
from sklearn.isotonic import IsotonicRegression
from scipy.special import expit
SEEDS=[17286,17287,17288]
ROUNDS=20; CLIENTS=5; LR=.10; CLIP=1.0
url='https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
try: data=pd.read_excel(url,header=1)
except Exception:
 from google.colab import files
 print('Upload original UCI credit default dataset')
 data=pd.read_excel(next(iter(files.upload())),header=1)
data.columns=[str(c).strip() for c in data.columns]
X=data.drop(columns=['ID','default payment next month'],errors='ignore').to_numpy(dtype=float)
y=data['default payment next month'].to_numpy(dtype=int)
assert len(y)==30000
def loss(p,labels):
 p=np.clip(p,1e-8,1-1e-8)
 return -(labels*np.log(p)+(1-labels)*np.log1p(-p))
def attack_metrics(score,truth):
 auc=roc_auc_score(truth,score); fpr,tpr,_=roc_curve(truth,score)
 return float(auc),float(np.max(tpr[fpr<=.01]))
def train_fed(X,y,client_indices,treatment,seed):
 rng=np.random.default_rng(seed);w=np.zeros(X.shape[1]+1)
 for rnd in range(ROUNDS):
  updates=[];sizes=[]
  for ids in client_indices:
   xi=np.column_stack([np.ones(len(ids)),X[ids]])
   g=xi.T@(expit(xi@w)-y[ids])/len(ids)
   if treatment!='D0':
    norm=np.linalg.norm(g);g=g*min(1.,CLIP/(norm+1e-12))
   updates.append(g);sizes.append(len(ids))
  g=np.average(updates,axis=0,weights=sizes)
  if treatment=='D2':
   # Experimental perturbation only; NOT a formally accounted DP guarantee.
   g=g+rng.normal(0,.05*CLIP/CLIENTS,size=g.shape)
  w=w-LR*g
 return w
rows=[]
for seed in SEEDS:
 rng=np.random.default_rng(seed)
 idx=np.arange(len(y))
 trainpool,test=train_test_split(idx,test_size=.2,stratify=y,random_state=seed)
 train,hold=train_test_split(trainpool,test_size=.25,stratify=y[trainpool],random_state=seed+1)
 train,cal=train_test_split(train,test_size=.2,stratify=y[train],random_state=seed+2)
 scaler=StandardScaler().fit(X[train]);xx=scaler.transform(X)
 for cond in ('C1_iid','C2_imbalanced'):
  client_idx=[]
  if cond=='C1_iid':
   perm=rng.permutation(train);client_idx=[a for a in np.array_split(perm,CLIENTS)]
  else:
   # Deterministic label-skew assignment with every client retaining both classes.
   zero=rng.permutation(train[y[train]==0]);one=rng.permutation(train[y[train]==1])
   weights=np.array([.02,.08,.15,.25,.50])
   chunks0=np.split(zero,np.cumsum(np.round(weights[:-1]*len(zero)).astype(int)))
   chunks1=np.split(one,np.cumsum(np.round(weights[::-1][:-1]*len(one)).astype(int)))
   client_idx=[np.r_[a,b] for a,b in zip(chunks0,chunks1)]
  for treatment in ('D0','D0C','D2'):
   w=train_fed(xx,y,client_idx,treatment,seed)
   prob=lambda ids:expit(np.column_stack([np.ones(len(ids)),xx[ids]])@w)
   ptest=prob(test);pcal=prob(cal)
   iso=IsotonicRegression(out_of_bounds='clip').fit(pcal,y[cal])
   pcal_test=iso.predict(ptest)
   m=[];n=[]
   for cls in (0,1):
    mi=train[y[train]==cls];ni=hold[y[hold]==cls];k=min(len(mi),len(ni))
    m.extend(rng.choice(mi,k,replace=False));n.extend(rng.choice(ni,k,replace=False))
   cand=np.r_[m,n];membership=np.r_[np.ones(len(m),int),np.zeros(len(n),int)]
   score=-loss(prob(cand),y[cand])
   mia_auc,tpr=attack_metrics(score,membership)
   rows.append(dict(seed=seed,condition=cond,treatment=treatment,n_train=len(train),rounds=ROUNDS,clients=CLIENTS,
     utility_auc=roc_auc_score(y[test],ptest),brier=brier_score_loss(y[test],ptest),
     isotonic_brier=brier_score_loss(y[test],pcal_test),isotonic_auc=roc_auc_score(y[test],pcal_test),
     matched_attack_auc=mia_auc,matched_tpr_at_1pct_fpr=tpr,n_attack=len(cand),
     privacy_accounted=False))
out=pd.DataFrame(rows)
print(out.to_string(index=False))
print('\nMEANS BY CONDITION/TREATMENT\n',out.groupby(['condition','treatment']).mean(numeric_only=True).to_string())
out.to_csv('/content/paper3_federated_pilot_results.csv',index=False)
print('\nThis is a reconstructed, 3-seed, 20-round experimental pilot. D2 is heuristic perturbation, NOT certified differential privacy. C1/C2 are illustrative and not verified original paper definitions. No formal privacy epsilon is claimed.')